<a href="https://colab.research.google.com/github/cerenvar/meg-word-decoding/blob/ceren/notebooks/ip3_hibrit_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/cerenvar/meg-word-decoding/blob/main/notebooks/ip3_hibrit_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# İP-3 | CNN–Transformer Hibrit Model (M2) ve Parametre Eşleştirilmiş Kontrol (E5)

**Proje:** MEG Sinyallerinden Dinlenen Kelimenin Çözümlenmesi: Kişiye Uyarlanabilir ve Veri-Verimli Bir Derin Öğrenme Yaklaşımı (TÜBİTAK 2209-A)  
**İş Paketi:** İP-3

Bu not defteri, İP-1'in pencere arşivi ve manifesti ile İP-2'nin veri yükleyici, kayıp fonksiyonu ve değerlendirme düzenini **değiştirmeden** kullanır; böylece M2 ve E5, İP-2'deki M1 ile aynı koşullarda karşılaştırılır.

| Çıktı | Bölüm | Dosya (`MEG_2209A/ip3/`) |
| :--- | :--- | :--- |
| Ortam bilgisi, İP-1 ve İP-2 doğrulaması | Bölüm 1 | `run_info.json` |
| Ortak veri yükleyici (İP-2'den aynen) | Bölüm 2 | — |
| Kayıp fonksiyonu ve metrikler (İP-2'den aynen) | Bölüm 3 | — |
| M1 yeniden üretim kontrolü | Bölüm 3.1 | `m1_reproduction.json` |
| M2 ve E5 mimarileri, parametre eşleştirme | Bölüm 4.1 | `model_configs.json` |
| Model süre, bellek ve gecikme ön ölçümü | Bölüm 4.2 | `model_benchmark.csv` |
| Ortak eğitim düzeni ve deneme koşusu (seed 2209) | Bölüm 5 | `runs/w500/seed2209/` |
| Pencere uzunluğu, çoklu tohum, kontroller | Sonraki bölümler | eklenecek |
| Öneriden sapma kaydı | Tüm bölümler | `deviations.csv` |

**Çalıştırma kuralı:** `MEG_2209A/ip1` ve `MEG_2209A/ip2` yalnızca okunur. Tüm İP-3 çıktıları `MEG_2209A/ip3` klasörüne yazılır. Değerlendirme katılımcılarının 12. oturumu (`eval_test`) hiçbir seçimde kullanılmaz; mimari ve ayar seçimleri yalnızca `src_validation` üzerinden yapılır.

### Çalıştırma notları

1. *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin.
2. Her yeni oturumda Bölüm 1–3 çalıştırılır. Bölüm 1, ~28 GB'lık pencere arşivini Colab'ın yerel diskine kopyalar (ilk seferde ~13 dk); kopyalama yarıda kalırsa kaldığı yerden devam eder.
3. Kopyalama veya kayıt sırasında Drive bağlantısı koparsa kod kendiliğinden yeniden bağlanmayı dener (izin penceresi çıkarsa onaylayın). Elle yeniden bağlamak gerekirse:
   ```python
   from google.colab import drive
   drive.flush_and_unmount()
   drive.mount("/content/drive", force_remount=True)
   ```
4. Uzun eğitimler (Bölüm 5) her epoch sonunda `ip3/runs/` altına kayıt alır. Oturum koparsa **Tümünü çalıştır** yeterlidir: tamamlanmış tohumlar atlanır, yarım kalan tohum son epoch'tan devam eder.


## Bölüm 1. Ortam, İP-1 / İP-2 Doğrulama ve Yerel Arşiv Hazırlığı

İP-1 ve İP-2 ile aynı kütüphane sürümü (`pnpl==0.2.0`), aynı tohum (`SEED = 2209`) ve aynı veri sürümü (`HF_REVISION`) kullanılır. İP-1'in 8 sızıntı testinin geçtiği ve İP-2'nin M1 kontrol noktasının (`m1_best.pt`) mevcut olduğu doğrulanır.

In [ ]:
%pip install -q "pnpl==0.2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import gc
import json
import os
import subprocess
import platform
import random
import shutil
import sys
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, IterableDataset
from tqdm.auto import tqdm

BASE = Path("/content/drive/MyDrive/MEG_2209A")
IP1 = BASE / "ip1"   # salt okunur
IP2 = BASE / "ip2"   # salt okunur
OUT = BASE / "ip3"   # İP-3 çıktıları
OUT.mkdir(parents=True, exist_ok=True)

SEED = 2209
WINDOW_SEC = 0.5
EXPECTED_SFREQ = 250.0
N_CHANNELS = 306
N_CLASSES = 50
N_POINTS = int(WINDOW_SEC * EXPECTED_SFREQ)

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("UYARI: GPU bulunamadı. Çalışma zamanı türünü T4 GPU yapın.")

# İP-1 doğrulaması
assert (IP1 / "leakage_tests.csv").exists(), "HATA: MEG_2209A/ip1 bulunamadı! Drive kısayolunu kontrol edin."
leak = pd.read_csv(IP1 / "leakage_tests.csv")
assert leak["result"].eq("PASS").all(), "HATA: İP-1 sızıntı testlerinde başarısız test var!"

# İP-2 doğrulaması (M1 referans modeli)
M1_CKPT = IP2 / "m1_best.pt"
assert M1_CKPT.exists(), "HATA: MEG_2209A/ip2/m1_best.pt bulunamadı!"

run_info_ip1 = json.loads((IP1 / "run_info.json").read_text())
HF_REVISION = run_info_ip1["hf_revision"]

manifest = pd.read_parquet(IP1 / "window_manifest.parquet")
norm_data = np.load(IP1 / "normalization_subj0_train.npz")
NORM_MEAN = norm_data["mean"].astype(np.float32)
NORM_STD = norm_data["std"].astype(np.float32)
roles = json.loads((IP1 / "participant_roles.json").read_text())
class_freq = pd.read_csv(IP1 / "class_frequencies.csv")
VOCAB = class_freq["word"].tolist()

assert len(VOCAB) == N_CLASSES
assert NORM_MEAN.shape == (N_CHANNELS,) and NORM_STD.shape == (N_CHANNELS,)
assert not np.isnan(NORM_MEAN).any() and (NORM_STD > 0).all()

# Öneriden sapma kaydı (İP-3'e ait)
DEVIATIONS = {}
def log_deviation(key, section, proposal_says, decision, reason):
    DEVIATIONS[key] = {
        "section": section,
        "proposal": proposal_says,
        "implementation": decision,
        "rationale": reason,
    }
    pd.DataFrame(DEVIATIONS.values()).to_csv(OUT / "deviations.csv", index=False)

run_info_ip3 = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "device": torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu",
    "pnpl": version("pnpl"),
    "torch": version("torch"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "hf_revision": HF_REVISION,
    "seed": SEED,
    "window_sec": WINDOW_SEC,
}
(OUT / "run_info.json").write_text(json.dumps(run_info_ip3, indent=2, ensure_ascii=False))

print(f"İP-1 ve İP-2 doğrulaması tamamlandı | Cihaz: {run_info_ip3['device']}")
print(f"Toplam arşiv penceresi: {len(manifest):,} | src_train: {(manifest['split_role'] == 'src_train').sum():,}")

# Pencere arşivini yerel SSD'ye kopyala (eksik veya yarım dosyaları tamamlar)
LOCAL_ARCHIVE = Path("/content/windows_500ms_local")
LOCAL_ARCHIVE.mkdir(exist_ok=True)
drive_files = sorted((BASE / "windows_500ms_pT").glob("*.npy"))
assert len(drive_files) == 226, f"Arşivde 226 dosya bekleniyordu, {len(drive_files)} bulundu."

def remount_drive():
    """Kopan Drive bağlantısını yeniden kurar (izin penceresi çıkabilir)."""
    try:
        drive.flush_and_unmount()
    except Exception:
        subprocess.run(["fusermount", "-u", "/content/drive"], capture_output=True)
    drive.mount("/content/drive", force_remount=True)

def copy_with_retry(src: Path, dst: Path, retries: int = 5) -> bool:
    for attempt in range(1, retries + 1):
        try:
            if dst.exists() and dst.stat().st_size == src.stat().st_size:
                return False
            shutil.copy2(src, dst)
            return True
        except OSError as e:
            print(f"\n{src.name}: Drive bağlantı hatası ({e.strerror}); yeniden bağlanılıyor ({attempt}/{retries})")
            time.sleep(5)
            remount_drive()
    raise RuntimeError("Drive bağlantısı tekrar tekrar koptu. Çalışma zamanını yeniden başlatıp Tümünü çalıştır deneyin.")

copied = 0
for src in tqdm(drive_files, desc="Arşiv yerel SSD'ye kopyalanıyor"):
    copied += copy_with_retry(src, LOCAL_ARCHIVE / src.name)

print(f"Yerel arşiv hazır: {len(drive_files)} dosya (yeni kopyalanan: {copied})")

İP-1 ve İP-2 doğrulaması tamamlandı | Cihaz: Tesla T4
Toplam arşiv penceresi: 364,643 | src_train: 307,329


Arşiv yerel SSD'ye kopyalanıyor:   0%|          | 0/226 [00:00<?, ?it/s]


sub-0_ses-10_task-Sherlock2_run-1.npy: Drive bağlantı hatası (Transport endpoint is not connected); yeniden bağlanılıyor (1/5)
Mounted at /content/drive
Yerel arşiv hazır: 226 dosya (yeni kopyalanan: 226)


## Bölüm 2. Ortak Veri Yükleyici

İP-2 Bölüm 2'deki `ChunkedMEGDataset` **değiştirilmeden** kullanılır (bellek eşlemeli blok okuma, `chunk_size=2048`, `buffer_size=4096`). Okuma hızı İP-2'de ölçüldüğü için ön ölçüm tekrarlanmaz. Normalizasyon GPU üzerinde batch düzeyinde uygulanır (Bölüm 3).

In [ ]:
class ChunkedMEGDataset(IterableDataset):
    """
    İP-2 Bölüm 2'den aynen alınmıştır.
    Kayıt dosyalarını küçük sıralı bloklar halinde okuyup sınırlı RAM tamponunda karıştırır.
    """
    def __init__(self, df: pd.DataFrame, archive_dir: Path, shuffle: bool = True,
                 buffer_size: int = 4096, chunk_size: int = 2048, seed: int = SEED):
        super().__init__()
        self.archive_dir = Path(archive_dir)
        self.shuffle = shuffle
        self.buffer_size = buffer_size
        self.chunk_size = chunk_size
        self.seed = seed
        self.epoch = 0

        self.records = []
        for rec_file, grp in df.groupby("archive_file", sort=True):
            grp_sorted = grp.sort_values("archive_row")
            self.records.append({
                "file": rec_file,
                "rows": grp_sorted["archive_row"].to_numpy(dtype=np.int64),
                "labels": grp_sorted["label"].to_numpy(dtype=np.int64),
            })
        self.total_samples = len(df)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.total_samples

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        recs = list(self.records)

        if self.shuffle:
            rng = np.random.default_rng(self.seed + self.epoch)
            rng.shuffle(recs)

        if worker_info is not None:
            recs = recs[worker_info.id :: worker_info.num_workers]
            if self.shuffle:
                rng = np.random.default_rng(self.seed + self.epoch * 100 + worker_info.id)

        if not self.shuffle:
            for rec in recs:
                rows = rec["rows"]
                labels = rec["labels"]
                if len(rows) == 0:
                    continue
                arr = np.load(self.archive_dir / rec["file"], mmap_mode="r")
                for start in range(0, len(rows), self.chunk_size):
                    end = min(start + self.chunk_size, len(rows))
                    data = np.asarray(arr[rows[start:end]], dtype=np.float32)
                    sub_labels = labels[start:end]
                    for i in range(len(sub_labels)):
                        yield torch.from_numpy(data[i].copy()), int(sub_labels[i])
                    del data
            return

        buf_x, buf_y = [], []
        for rec in recs:
            rows = rec["rows"]
            labels = rec["labels"]
            if len(rows) == 0:
                continue
            arr = np.load(self.archive_dir / rec["file"], mmap_mode="r")

            for start in range(0, len(rows), self.chunk_size):
                end = min(start + self.chunk_size, len(rows))
                data = np.asarray(arr[rows[start:end]], dtype=np.float32)
                sub_labels = labels[start:end]

                perm = rng.permutation(len(sub_labels))
                data = data[perm]
                sub_labels = sub_labels[perm]

                for i in range(len(sub_labels)):
                    if len(buf_x) < self.buffer_size:
                        buf_x.append(data[i].copy())
                        buf_y.append(int(sub_labels[i]))
                    else:
                        idx = int(rng.integers(0, self.buffer_size))
                        out_x, out_y = buf_x[idx], buf_y[idx]
                        buf_x[idx] = data[i].copy()
                        buf_y[idx] = int(sub_labels[i])
                        yield torch.from_numpy(out_x), out_y

                del data, perm

        if len(buf_x) > 0:
            rem_perm = rng.permutation(len(buf_x))
            for idx in rem_perm:
                yield torch.from_numpy(buf_x[idx]), buf_y[idx]
        buf_x.clear()
        buf_y.clear()
        gc.collect()


train_df = manifest.loc[manifest["split_role"] == "src_train"].reset_index(drop=True)
val_df = manifest.loc[manifest["split_role"] == "src_validation"].reset_index(drop=True)
test_df = manifest.loc[manifest["split_role"] == "src_test"].reset_index(drop=True)

train_ds = ChunkedMEGDataset(train_df, LOCAL_ARCHIVE, shuffle=True, buffer_size=4096, chunk_size=2048, seed=SEED)
train_loader = DataLoader(train_ds, batch_size=256, num_workers=1, pin_memory=False, prefetch_factor=2)

val_ds = ChunkedMEGDataset(val_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED)
val_loader = DataLoader(val_ds, batch_size=512, num_workers=1, pin_memory=False)

test_ds = ChunkedMEGDataset(test_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED)
test_loader = DataLoader(test_ds, batch_size=512, num_workers=1, pin_memory=False)

print(f"src_train: {len(train_df):,} | src_validation: {len(val_df):,} | src_test: {len(test_df):,}")

src_train: 307,329 | src_validation: 3,052 | src_test: 2,681


## Bölüm 3. Kayıp Fonksiyonu, Metrikler ve Değerlendirme

İP-2 Bölüm 3–4'teki düzen **değiştirilmeden** kullanılır:

* **Kayıp:** ters karekök frekans ağırlıklı `CrossEntropyLoss` (ağırlıklar yalnız `src_train` sınıf sayılarından).
* **Metrikler:** `BAcc@1/5/10` (sınıf başına recall ortalaması), `Macro-F1`, `Top-1 Accuracy`. Temel ölçüt `BAcc@10`; şans düzeyi 10/50 = 0,20.
* **Normalizasyon:** İP-1'in `src_train` kanal ortalaması ve standart sapması, GPU üzerinde batch düzeyinde.

In [ ]:
from sklearn.metrics import f1_score

# Sınıf ağırlıkları: yalnız src_train sınıf sayılarından (İP-2 ile aynı)
train_counts = train_df["label"].value_counts().reindex(range(N_CLASSES), fill_value=0).to_numpy(dtype=np.float64)
class_weights = 1.0 / (np.sqrt(train_counts) + 1e-8)
class_weights[train_counts == 0] = 0
class_weights = class_weights / class_weights.sum() * N_CLASSES
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

mean_t = torch.from_numpy(NORM_MEAN).to(device).view(1, N_CHANNELS, 1)
std_t = torch.from_numpy(NORM_STD).to(device).view(1, N_CHANNELS, 1)

def normalize_batch(bx: torch.Tensor) -> torch.Tensor:
    bx = bx.to(device, non_blocking=True).float()
    return (bx - mean_t) / std_t

def compute_metrics(logits_or_probs: np.ndarray, y_true: np.ndarray, n_classes: int = N_CLASSES) -> dict:
    order = np.argsort(-logits_or_probs, axis=1)
    hits_1 = (order[:, :1] == y_true[:, None]).any(axis=1).astype(np.float64)
    hits_5 = (order[:, :5] == y_true[:, None]).any(axis=1).astype(np.float64)
    hits_10 = (order[:, :10] == y_true[:, None]).any(axis=1).astype(np.float64)

    recall_1, recall_5, recall_10 = [], [], []
    valid_classes = 0
    for c in range(n_classes):
        mask = (y_true == c)
        if mask.sum() > 0:
            recall_1.append(hits_1[mask].mean())
            recall_5.append(hits_5[mask].mean())
            recall_10.append(hits_10[mask].mean())
            valid_classes += 1
        else:
            recall_1.append(np.nan)
            recall_5.append(np.nan)
            recall_10.append(np.nan)

    pred_classes = np.argmax(logits_or_probs, axis=1)
    macro_f1 = float(f1_score(y_true, pred_classes, average="macro"))

    return {
        "bacc@1": float(np.nanmean(recall_1)),
        "bacc@5": float(np.nanmean(recall_5)),
        "bacc@10": float(np.nanmean(recall_10)),
        "macro_f1": macro_f1,
        "top1_acc": float(hits_1.mean()),
        "valid_classes": valid_classes,
    }

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    all_logits, all_targets = [], []
    total_loss, total_n = 0.0, 0
    for bx, by in loader:
        x_norm = normalize_batch(bx)
        by_dev = by.to(device, non_blocking=True).long()
        logits = model(x_norm)
        loss = criterion(logits, by_dev)
        total_loss += float(loss.item()) * bx.size(0)
        total_n += bx.size(0)
        all_logits.append(logits.cpu().numpy())
        all_targets.append(by.numpy())

    metrics = compute_metrics(np.concatenate(all_logits), np.concatenate(all_targets))
    metrics["loss"] = total_loss / max(total_n, 1)
    metrics["n_samples"] = int(total_n)
    return metrics

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

ratio = train_counts.max() / train_counts[train_counts > 0].min()
print(f"Sınıf dengesizlik oranı: {ratio:.2f}x | Kayıp: ters karekök ağırlıklı CrossEntropyLoss")

Sınıf dengesizlik oranı: 121.24x | Kayıp: ters karekök ağırlıklı CrossEntropyLoss


### Bölüm 3.1. M1 Yeniden Üretim Kontrolü

M2 ve E5'in M1 ile adil karşılaştırılabilmesi için İP-3'ün veri hattı, normalizasyonu ve metrikleri İP-2 ile birebir aynı olmalıdır. Bunu doğrulamak için İP-2'nin `m1_best.pt` kontrol noktası yüklenip `src_test` üzerinde yeniden değerlendirilir; sonuç İP-2'de raporlanan değerlerle (`BAcc@10 = 0,2649`, `Top-1 = 0,1380`) eşleşmelidir. Bu adımda model eğitilmez.

In [ ]:
class TemporalCNN_M1(nn.Module):
    """İP-2 Bölüm 5'ten aynen alınmıştır (kontrol noktasının yüklenebilmesi için katman yapısı değiştirilmemelidir)."""
    def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(32, 64, kernel_size=5, padding=2, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

IP2_REPORTED = {"bacc@10": 0.2649, "top1_acc": 0.1380}
TOLERANCE = 0.002

m1 = TemporalCNN_M1().to(device)
ckpt = torch.load(M1_CKPT, map_location=device)
m1.load_state_dict(ckpt["model_state_dict"])
M1_PARAMS = count_params(m1)

t0 = time.perf_counter()
m1_test = evaluate_model(m1, test_loader)
elapsed = time.perf_counter() - t0

checks = {k: abs(m1_test[k] - v) <= TOLERANCE for k, v in IP2_REPORTED.items()}
m1_repro = {
    "checkpoint": str(M1_CKPT),
    "checkpoint_epoch": int(ckpt.get("epoch", -1)),
    "params": M1_PARAMS,
    "n_samples": m1_test["n_samples"],
    "test_bacc@10": round(m1_test["bacc@10"], 4),
    "test_bacc@5": round(m1_test["bacc@5"], 4),
    "test_bacc@1": round(m1_test["bacc@1"], 4),
    "test_macro_f1": round(m1_test["macro_f1"], 4),
    "test_top1_acc": round(m1_test["top1_acc"], 4),
    "test_loss": round(m1_test["loss"], 4),
    "ip2_reported": IP2_REPORTED,
    "tolerance": TOLERANCE,
    "matches_ip2": all(checks.values()),
    "elapsed_sec": round(elapsed, 2),
}
(OUT / "m1_reproduction.json").write_text(json.dumps(m1_repro, indent=2, ensure_ascii=False))

print(f"M1 parametre sayısı: {M1_PARAMS:,} | src_test pencere: {m1_test['n_samples']:,}")
print(f"BAcc@10 = {m1_test['bacc@10']:.4f} (İP-2: {IP2_REPORTED['bacc@10']}) | "
      f"Top-1 = {m1_test['top1_acc']:.4f} (İP-2: {IP2_REPORTED['top1_acc']})")
assert m1_repro["matches_ip2"], "HATA: M1 sonucu İP-2 ile eşleşmiyor; veri hattı veya metrikler farklı olabilir."
print("M1 yeniden üretimi başarılı: İP-3 veri hattı İP-2 ile aynı.")

M1 parametre sayısı: 110,258 | src_test pencere: 2,681
BAcc@10 = 0.2649 (İP-2: 0.2649) | Top-1 = 0.1380 (İP-2: 0.138)
M1 yeniden üretimi başarılı: İP-3 veri hattı İP-2 ile aynı.


## Bölüm 4. M2 ve E5 Mimarileri, Parametre Eşleştirme ve Ön Ölçüm

**4.1 Mimariler.** M2 (CNN–Transformer) önerideki başlangıç yapılandırmasıyla kurulur: 2 evrişim bloğu, 128 boyutlu gömme, 2 Transformer kodlayıcı katmanı, 4 dikkat başlığı, ileri besleme 256, dropout 0,1. E5, M1 mimarisinin kanal genişlikleri (32/64/128 oranı korunarak) M2'nin parametre sayısına %5'ten az farkla eşitlenmiş hâlidir. Üç modelin gerçek bir doğrulama batch'inde doğru boyutta çıktı ürettiği denetlenir; yapılandırmalar `model_configs.json` dosyasına kaydedilir.

**4.2 Ön ölçüm.** Üç model için eğitim adımı süresi, tepe GPU belleği ve çıkarım gecikmesi ölçülür; İP-2'deki veri okuma süresiyle birlikte epoch ve kaynak eğitim süresi tahmin edilir (`model_benchmark.csv`). M2'nin hesaplama süresi veri okumayı aşarsa R3'teki optimizasyonlar değerlendirilir.

In [ ]:
# Eğitim adımlarında tekrar üretilebilirlik için deterministik cuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class HybridCNNTransformer_M2(nn.Module):
    """
    M2: 2 evrişim bloğu (yerel zamansal-uzamsal özellikler) + Transformer kodlayıcı (uzun erimli ilişkiler).
    Başlangıç yapılandırması: 128 boyutlu gömme, 2 kodlayıcı katmanı, 4 dikkat başlığı, ileri besleme 256, dropout 0,1.
    downsample > 1 verilirse Transformer'dan önce zamansal alt örnekleme yapılır (yalnızca bellek/süre gerekirse).
    """
    def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, n_points: int = N_POINTS,
                 conv_channels: int = 64, d_model: int = 128, n_heads: int = 4, n_layers: int = 2,
                 ff_dim: int = 256, dropout: float = 0.1, downsample: int = 1):
        super().__init__()
        self.conv = nn.Sequential(
            # Blok 1: 64 kanal, çekirdek 7
            nn.Conv1d(in_channels, conv_channels, kernel_size=7, padding=3, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=conv_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            # Blok 2: 128 kanal (gömme boyutu), çekirdek 5
            nn.Conv1d(conv_channels, d_model, kernel_size=5, padding=2, stride=downsample, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        n_tokens = (n_points - 1) // downsample + 1
        self.pos_emb = nn.Parameter(torch.zeros(1, n_tokens, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
                                           dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.conv(x).transpose(1, 2)        # (B, zaman, d_model)
        h = self.encoder(h + self.pos_emb)
        h = self.norm(h.mean(dim=1))            # zamansal ortalama havuzlama (M1 ile aynı okuma biçimi)
        return self.head(h)


class ParamMatchedCNN_E5(nn.Module):
    """E5: M1 mimarisi (3 blok, çekirdek 7/5/3, GroupNorm, GELU, dropout 0,2); yalnızca kanal genişlikleri büyütülmüştür."""
    def __init__(self, widths, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, dropout: float = 0.2):
        super().__init__()
        c1, c2, c3 = widths
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, c1, kernel_size=7, padding=3, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=c1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(c1, c2, kernel_size=5, padding=2, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=c2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(c2, c3, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=c3),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(c3, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# 1) M2'nin parametre sayısı
M2_CONFIG = dict(conv_channels=64, d_model=128, n_heads=4, n_layers=2, ff_dim=256, dropout=0.1, downsample=1)
M2_PARAMS = count_params(HybridCNNTransformer_M2(**M2_CONFIG))

# 2) E5 genişliklerinin aranması: M1'in 32/64/128 oranı korunur, genişlikler 8'in katına yuvarlanır (GroupNorm için)
def m1_family_params(c1, c2, c3, in_channels=N_CHANNELS, n_classes=N_CLASSES):
    return (in_channels * 7 * c1 + 2 * c1) + (c1 * 5 * c2 + 2 * c2) + (c2 * 3 * c3 + 2 * c3) + (c3 * n_classes + n_classes)

assert m1_family_params(32, 64, 128) == M1_PARAMS, "Parametre formülü M1 ile tutarsız"

best = None
for scale in np.arange(1.0, 6.0, 0.001):
    widths = tuple(max(8, int(8 * round(base * scale / 8))) for base in (32, 64, 128))
    diff = abs(m1_family_params(*widths) - M2_PARAMS) / M2_PARAMS
    if best is None or diff < best[0]:
        best = (diff, widths)
E5_WIDTHS = best[1]
E5_PARAMS = count_params(ParamMatchedCNN_E5(E5_WIDTHS))
assert E5_PARAMS == m1_family_params(*E5_WIDTHS)
param_diff = abs(E5_PARAMS - M2_PARAMS) / M2_PARAMS
assert param_diff < 0.05, f"E5–M2 parametre farkı %5'i aşıyor: {param_diff:.2%}"

# 3) Gerçek bir doğrulama batch'i ile çıktı boyutu kontrolü
seed_everything(SEED)
check_models = {
    "M1": TemporalCNN_M1().to(device),
    "E5": ParamMatchedCNN_E5(E5_WIDTHS).to(device),
    "M2": HybridCNNTransformer_M2(**M2_CONFIG).to(device),
}
bx, by = next(iter(val_loader))
with torch.no_grad():
    for name, model in check_models.items():
        model.eval()
        out = model(normalize_batch(bx))
        assert out.shape == (bx.size(0), N_CLASSES), f"{name} çıktı boyutu hatalı: {tuple(out.shape)}"
        assert torch.isfinite(out).all(), f"{name} çıktısında NaN/Inf var"
del check_models, bx, by
torch.cuda.empty_cache()

# 4) Yapılandırmaların kaydı
model_configs = {
    "M1": {"widths": [32, 64, 128], "kernels": [7, 5, 3], "dropout": 0.2, "params": M1_PARAMS},
    "E5": {"widths": list(E5_WIDTHS), "kernels": [7, 5, 3], "dropout": 0.2, "params": E5_PARAMS,
           "matched_to": "M2", "param_diff_vs_M2": round(param_diff, 6)},
    "M2": {**M2_CONFIG, "conv_kernels": [7, 5], "positional_embedding": "learned", "norm_first": True,
           "pooling": "temporal_mean", "params": M2_PARAMS},
}
(OUT / "model_configs.json").write_text(json.dumps(model_configs, indent=2, ensure_ascii=False))

print(f"M1 parametre: {M1_PARAMS:,}")
print(f"M2 parametre: {M2_PARAMS:,}")
print(f"E5 parametre: {E5_PARAMS:,} | genişlikler {E5_WIDTHS} | M2'ye göre fark: %{param_diff * 100:.3f}")
print("Çıktı boyutu kontrolü geçti: üç model de (batch, 50) logit üretiyor.")

M1 parametre: 110,258
M2 parametre: 466,098
E5 parametre: 466,034 | genişlikler (88, 176, 344) | M2'ye göre fark: %0.014
Çıktı boyutu kontrolü geçti: üç model de (batch, 50) logit üretiyor.


In [ ]:
import math

def benchmark_model(model: nn.Module, batch_size: int = 256, n_warmup: int = 5, n_steps: int = 30) -> dict:
    """Rastgele girdiyle eğitim adımı süresi, tepe GPU belleği ve çıkarım gecikmesi (veri okuma hariç)."""
    model = model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    x = torch.randn(batch_size, N_CHANNELS, N_POINTS, device=device)
    y = torch.randint(0, N_CLASSES, (batch_size,), device=device)

    torch.cuda.reset_peak_memory_stats()
    for i in range(n_warmup + n_steps):
        if i == n_warmup:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
    torch.cuda.synchronize()
    train_ms = (time.perf_counter() - t0) / n_steps * 1000
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3

    model.eval()
    with torch.no_grad():
        xb = torch.randn(512, N_CHANNELS, N_POINTS, device=device)
        for _ in range(3):
            model(xb)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(10):
            model(xb)
        torch.cuda.synchronize()
        batch_ms_per_sample = (time.perf_counter() - t0) / (10 * 512) * 1000

        x1 = torch.randn(1, N_CHANNELS, N_POINTS, device=device)
        for _ in range(10):
            model(x1)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(50):
            model(x1)
        torch.cuda.synchronize()
        single_ms = (time.perf_counter() - t0) / 50 * 1000

    return {"train_ms_per_step": train_ms, "peak_gpu_gb": peak_gb,
            "infer_ms_per_sample_batch512": batch_ms_per_sample, "infer_ms_single": single_ms}


assert device.type == "cuda", "Ön ölçüm GPU gerektirir: çalışma zamanı türünü T4 GPU yapın."

# İP-2'de ölçülen epoch başına veri okuma süresi
try:
    READ_MIN_PER_EPOCH = json.loads((IP2 / "dataloader_benchmark.json").read_text())["epoch_read_min"]
except Exception:
    READ_MIN_PER_EPOCH = 3.67
STEPS_PER_EPOCH = math.ceil(len(train_df) / 256)

bench_rows = []
for name, build in [("M1", lambda: TemporalCNN_M1()),
                    ("E5", lambda: ParamMatchedCNN_E5(E5_WIDTHS)),
                    ("M2", lambda: HybridCNNTransformer_M2(**M2_CONFIG))]:
    seed_everything(SEED)
    model = build()
    r = benchmark_model(model)
    compute_min = r["train_ms_per_step"] * STEPS_PER_EPOCH / 60000
    epoch_min = max(compute_min, READ_MIN_PER_EPOCH)   # okuma ve hesaplama örtüşür; yavaş olan belirler
    bench_rows.append({
        "model": name,
        "params": count_params(model),
        "train_ms_per_step_b256": round(r["train_ms_per_step"], 2),
        "compute_min_per_epoch": round(compute_min, 2),
        "read_min_per_epoch": READ_MIN_PER_EPOCH,
        "est_min_per_epoch": round(epoch_min, 2),
        "est_hours_30_epochs": round(epoch_min * 30 / 60, 2),
        "peak_gpu_gb": round(r["peak_gpu_gb"], 3),
        "infer_ms_per_sample_batch512": round(r["infer_ms_per_sample_batch512"], 4),
        "infer_ms_single": round(r["infer_ms_single"], 3),
    })
    del model
    torch.cuda.empty_cache()

bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv(OUT / "model_benchmark.csv", index=False)
display(bench_df)

m2_row = bench_df.set_index("model").loc["M2"]
if m2_row["compute_min_per_epoch"] <= READ_MIN_PER_EPOCH:
    print("Sonuç: M2'de de darboğaz veri okuma. Karma duyarlık, gradyan biriktirme veya alt örnekleme gerekmiyor.")
else:
    print("Sonuç: M2'de hesaplama veri okumadan uzun. R3 seçenekleri (karma duyarlık / alt örnekleme) değerlendirilmeli.")

,model,params,train_ms_per_step_b256,compute_min_per_epoch,read_min_per_epoch,est_min_per_epoch,est_hours_30_epochs,peak_gpu_gb,infer_ms_per_sample_batch512,infer_ms_single
0,M1,110258,13.36,0.27,3.67,3.67,1.83,0.158,0.0108,0.786
1,E5,466034,30.97,0.62,3.67,3.67,1.83,0.337,0.0323,0.670
2,M2,466098,60.72,1.22,3.67,3.67,1.83,0.633,0.0553,1.113


Sonuç: M2'de de darboğaz veri okuma. Karma duyarlık, gradyan biriktirme veya alt örnekleme gerekmiyor.


## Bölüm 5. Ortak Eğitim Düzeni ve Deneme Koşusu

**5.1 Birlikte eğitim.** Ön ölçüme göre epoch süresini veri okuma belirliyor (~3,7 dk), üç modelin toplam hesaplaması ise bunun altında (~2,1 dk). Bu nedenle M1, E5 ve M2 tek bir veri akışıyla birlikte eğitilir: her batch diskten bir kez okunur ve üç modele aynı sırayla verilir. Her model kendi erken durdurma kararını bağımsız verir. Eğitim ayarları İP-2'deki M1 ile aynıdır (AdamW, lr 3e-4, wd 1e-2, gradyan kırpma 1,0, en fazla 30 epoch, doğrulama BAcc@10 üzerinden sabır 5); üç model aynı epoch bütçesi ve aynı tohum listesiyle eğitilir. Veri sırası da tohuma bağlıdır.

Her epoch sonunda model ağırlıkları, optimizer durumu, erken durdurma durumu ve rastgele sayı üreteçleri `ip3/runs/<pencere>/seed<tohum>/checkpoint.pt` dosyasına kaydedilir; oturum koparsa eğitim son tamamlanan epoch'tan devam eder. Her koşu sonunda en iyi doğrulama ağırlıklarıyla `src_test` değerlendirilir (betimsel) ve pencere bazlı test logitleri (`test_logits.npz`) süre grubu ve bootstrap analizleri için saklanır.

**5.2 Deneme koşusu.** Tohum listesinin ilk tohumuyla (2209) 500 ms pencerede üç model birlikte eğitilir. Amaç M2'nin öğrendiğini ve doğrulama oturumunda M1 ile kıyaslanabilir sonuç verdiğini görmektir. Ayarlar değişmezse bu koşu ana çoklu tohum deneylerinin ilk tohumu olarak kullanılır. Süre: ~2 saat.

In [ ]:
# Ortak eğitim ayarları: İP-2'deki M1 eğitimiyle aynı (AdamW, lr 3e-4, wd 1e-2, gradyan kırpma 1,0,
# en fazla 30 epoch, doğrulama BAcc@10 üzerinden sabır 5). Üç model için aynı epoch bütçesi ve aynı tohum listesi.
TRAIN_CFG = {
    "optimizer": "AdamW", "lr": 3e-4, "weight_decay": 1e-2, "grad_clip": 1.0,
    "max_epochs": 30, "patience": 5, "early_stop_metric": "val_bacc@10", "batch_size": 256,
}
SEEDS = [SEED + i for i in range(10)]   # 2209–2218; öncelik ilk 5 tohum
RUNS_DIR = OUT / "runs"
RUNS_DIR.mkdir(exist_ok=True)

MODEL_BUILDERS = {
    "M1": lambda: TemporalCNN_M1(),
    "E5": lambda: ParamMatchedCNN_E5(E5_WIDTHS),
    "M2": lambda: HybridCNNTransformer_M2(**M2_CONFIG),
}

# src_test pencerelerinin değerlendirme sırası (yükleyici kayıt dosyası ve satır sırasıyla okur)
TEST_ORDER = test_df.sort_values(["archive_file", "archive_row"]).reset_index(drop=True)
if not (RUNS_DIR / "src_test_order.parquet").exists():
    TEST_ORDER.to_parquet(RUNS_DIR / "src_test_order.parquet", index=False)


def save_to_drive(obj, path: Path, retries: int = 3):
    """Önce yerel diske, sonra Drive'a yazar; yarım dosya oluşmaması için geçici adla kopyalayıp yer değiştirir."""
    local_tmp = Path("/content/_save_tmp") / path.name
    local_tmp.parent.mkdir(exist_ok=True)
    torch.save(obj, local_tmp)
    for attempt in range(1, retries + 1):
        try:
            remote_tmp = path.with_name(path.name + ".tmp")
            shutil.copy2(local_tmp, remote_tmp)
            os.replace(remote_tmp, path)
            return
        except OSError as e:
            print(f"\nDrive'a yazılamadı ({e.strerror}); yeniden bağlanılıyor ({attempt}/{retries})")
            time.sleep(5)
            remount_drive()
    raise RuntimeError("Drive'a yazılamadı. Bu hücreyi tekrar çalıştırın; eğitim son kayıttan devam eder.")


@torch.no_grad()
def evaluate_members(models: dict, loader: DataLoader):
    """Veriyi bir kez okuyup birden fazla modeli aynı batch'lerle değerlendirir."""
    for m in models.values():
        m.eval()
    logits = {n: [] for n in models}
    loss_sum = {n: 0.0 for n in models}
    targets, total_n = [], 0
    for bx, by in loader:
        x = normalize_batch(bx)
        y = by.to(device, non_blocking=True).long()
        for n, m in models.items():
            out = m(x)
            loss_sum[n] += float(criterion(out, y).item()) * bx.size(0)
            logits[n].append(out.float().cpu().numpy())
        targets.append(by.numpy())
        total_n += bx.size(0)
    y_true = np.concatenate(targets)
    metrics = {}
    for n in models:
        logits[n] = np.concatenate(logits[n])
        metrics[n] = compute_metrics(logits[n], y_true)
        metrics[n]["loss"] = loss_sum[n] / max(total_n, 1)
    return metrics, logits, y_true


def run_joint_training(run_seed: int, member_names=("M1", "E5", "M2"), window_tag: str = "w500") -> dict:
    """
    Tek veri akışıyla birden fazla modeli birlikte eğitir: her batch bir kez okunur ve tüm aktif modellere verilir.
    Her model kendi erken durdurma kararını bağımsız verir. Her epoch sonunda Drive'a kayıt alınır;
    oturum koparsa hücre yeniden çalıştırıldığında son tamamlanan epoch'tan devam eder.
    """
    run_dir = RUNS_DIR / window_tag / f"seed{run_seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "results.json").exists():
        print(f"seed {run_seed} ({window_tag}) zaten tamamlanmış; sonuçlar kayıttan okunuyor.")
        return json.loads((run_dir / "results.json").read_text())

    ckpt_path = run_dir / "checkpoint.pt"
    seed_everything(run_seed)
    models = {n: MODEL_BUILDERS[n]().to(device) for n in member_names}
    opts = {n: torch.optim.AdamW(models[n].parameters(), lr=TRAIN_CFG["lr"], weight_decay=TRAIN_CFG["weight_decay"])
            for n in member_names}
    state = {n: {"best_val": -1.0, "best_epoch": 0, "bad_epochs": 0, "epochs_trained": 0, "stopped": False}
             for n in member_names}
    history, start_epoch, elapsed_prev = [], 1, 0.0

    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        for n in member_names:
            models[n].load_state_dict(ck["models"][n])
            opts[n].load_state_dict(ck["optims"][n])
        state, history = ck["state"], ck["history"]
        start_epoch, elapsed_prev = ck["epoch"] + 1, ck["elapsed_sec"]
        torch.set_rng_state(ck["rng"]["torch"])
        if torch.cuda.is_available():
            torch.cuda.set_rng_state_all(ck["rng"]["cuda"])
        np.random.set_state(ck["rng"]["numpy"])
        random.setstate(ck["rng"]["python"])
        print(f"Kayıt bulundu: seed {run_seed}, {ck['epoch']}. epoch tamamlanmış; {start_epoch}. epoch'tan devam ediliyor.")

    # Veri sırası da tohuma bağlı (seed 2209 için İP-2'deki sırayla aynı)
    run_train_ds = ChunkedMEGDataset(train_df, LOCAL_ARCHIVE, shuffle=True, buffer_size=4096, chunk_size=2048, seed=run_seed)
    run_train_loader = DataLoader(run_train_ds, batch_size=TRAIN_CFG["batch_size"], num_workers=1,
                                  pin_memory=False, prefetch_factor=2)

    t_session = time.perf_counter()
    for epoch in range(start_epoch, TRAIN_CFG["max_epochs"] + 1):
        active = [n for n in member_names if not state[n]["stopped"]]
        if not active:
            break
        run_train_ds.set_epoch(epoch)
        for n in active:
            models[n].train()
        loss_sum = {n: 0.0 for n in active}
        seen = 0
        t_epoch = time.perf_counter()

        for bx, by in tqdm(run_train_loader, total=STEPS_PER_EPOCH, leave=False,
                           desc=f"seed {run_seed} | epoch {epoch}/{TRAIN_CFG['max_epochs']}"):
            x = normalize_batch(bx)
            y = by.to(device, non_blocking=True).long()
            for n in active:
                opts[n].zero_grad()
                loss = criterion(models[n](x), y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(models[n].parameters(), max_norm=TRAIN_CFG["grad_clip"])
                opts[n].step()
                loss_sum[n] += float(loss.item()) * bx.size(0)
            seen += bx.size(0)

        val_metrics, _, _ = evaluate_members({n: models[n] for n in active}, val_loader)
        row = {"epoch": epoch, "epoch_min": round((time.perf_counter() - t_epoch) / 60, 2)}
        msgs = []
        for n in active:
            m = val_metrics[n]
            state[n]["epochs_trained"] = epoch
            row.update({
                f"{n}_train_loss": round(loss_sum[n] / max(seen, 1), 4),
                f"{n}_val_loss": round(m["loss"], 4),
                f"{n}_val_bacc@10": round(m["bacc@10"], 4),
                f"{n}_val_macro_f1": round(m["macro_f1"], 4),
            })
            if m["bacc@10"] > state[n]["best_val"]:
                state[n].update(best_val=m["bacc@10"], best_epoch=epoch, bad_epochs=0)
                save_to_drive({"epoch": epoch, "model_state_dict": models[n].state_dict(),
                               "val_bacc@10": m["bacc@10"], "seed": run_seed}, run_dir / f"{n}_best.pt")
            else:
                state[n]["bad_epochs"] += 1
                if state[n]["bad_epochs"] >= TRAIN_CFG["patience"]:
                    state[n]["stopped"] = True
            msgs.append(f"{n} val BAcc@10={m['bacc@10']:.4f}" + (" (durdu)" if state[n]["stopped"] else ""))
        history.append(row)

        save_to_drive({
            "epoch": epoch, "seed": run_seed, "train_cfg": TRAIN_CFG,
            "models": {n: models[n].state_dict() for n in member_names},
            "optims": {n: opts[n].state_dict() for n in member_names},
            "state": state, "history": history,
            "elapsed_sec": elapsed_prev + (time.perf_counter() - t_session),
            "rng": {"torch": torch.get_rng_state(),
                    "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                    "numpy": np.random.get_state(), "python": random.getstate()},
        }, ckpt_path)
        print(f"[seed {run_seed} | epoch {epoch}/{TRAIN_CFG['max_epochs']} | {row['epoch_min']:.1f} dk] " + " | ".join(msgs))

    # En iyi doğrulama ağırlıklarıyla src_test değerlendirmesi (betimsel; hiçbir seçim test verisiyle yapılmaz)
    for n in member_names:
        best = torch.load(run_dir / f"{n}_best.pt", map_location="cpu")
        models[n].load_state_dict(best["model_state_dict"])
    test_metrics, test_logits, y_test = evaluate_members(models, test_loader)
    assert np.array_equal(y_test, TEST_ORDER["label"].to_numpy()), "Test sırası manifestle uyuşmuyor"
    np.savez_compressed(Path("/content/_save_tmp") / "test_logits.npz", y_true=y_test, **test_logits)
    shutil.copy2(Path("/content/_save_tmp") / "test_logits.npz", run_dir / "test_logits.npz")

    total_sec = elapsed_prev + (time.perf_counter() - t_session)
    results = {"seed": run_seed, "window": window_tag, "train_cfg": TRAIN_CFG,
               "elapsed_hours": round(total_sec / 3600, 2), "models": {}}
    for n in member_names:
        tm = test_metrics[n]
        results["models"][n] = {
            "params": count_params(models[n]),
            "best_epoch": state[n]["best_epoch"],
            "epochs_trained": state[n]["epochs_trained"],
            "best_val_bacc@10": round(state[n]["best_val"], 4),
            "test_bacc@10": round(tm["bacc@10"], 4),
            "test_bacc@5": round(tm["bacc@5"], 4),
            "test_bacc@1": round(tm["bacc@1"], 4),
            "test_macro_f1": round(tm["macro_f1"], 4),
            "test_top1_acc": round(tm["top1_acc"], 4),
            "test_loss": round(tm["loss"], 4),
        }
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    (run_dir / "results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False))
    print(f"seed {run_seed} tamamlandı ({results['elapsed_hours']} saat).")
    return results


print(f"Eğitim ayarları: {TRAIN_CFG}")
print(f"Tohum listesi: {SEEDS}")
print(f"Kayıt klasörü: {RUNS_DIR}")

Eğitim ayarları: {'optimizer': 'AdamW', 'lr': 0.0003, 'weight_decay': 0.01, 'grad_clip': 1.0, 'max_epochs': 30, 'patience': 5, 'early_stop_metric': 'val_bacc@10', 'batch_size': 256}
Tohum listesi: [2209, 2210, 2211, 2212, 2213, 2214, 2215, 2216, 2217, 2218]
Kayıt klasörü: /content/drive/MyDrive/MEG_2209A/ip3/runs


In [ ]:
# Deneme koşusu: tohum listesinin ilk tohumu (2209), 500 ms pencere, M1 + E5 + M2 birlikte.
# Ayarlar değişmezse bu koşu ana çoklu tohum deneylerinin ilk tohumu olarak da kullanılır.
pilot = run_joint_training(SEEDS[0])

rows = []
for n, r in pilot["models"].items():
    rows.append({"model": n, "params": r["params"], "best_epoch": r["best_epoch"],
                 "val_bacc@10": r["best_val_bacc@10"], "test_bacc@10": r["test_bacc@10"],
                 "test_top1": r["test_top1_acc"], "test_macro_f1": r["test_macro_f1"]})
pilot_df = pd.DataFrame(rows)
display(pilot_df)
print(f"Karşılaştırma: İP-2'deki M1 test BAcc@10 = {IP2_REPORTED['bacc@10']} | şans düzeyi = 0.20")
print("Not: Model karşılaştırması doğrulama (val) sütununa göre yapılır; test sütunu yalnızca betimseldir.")

Kayıt bulundu: seed 2209, 9. epoch tamamlanmış; 10. epoch'tan devam ediliyor.


seed 2209 | epoch 10/30:   0%|          | 0/1201 [00:00<?, ?it/s]

[seed 2209 | epoch 10/30 | 5.1 dk] M1 val BAcc@10=0.2078 | E5 val BAcc@10=0.2340 | M2 val BAcc@10=0.2689


seed 2209 | epoch 11/30:   0%|          | 0/1201 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af8685a2160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af8685a2160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

[seed 2209 | epoch 11/30 | 5.0 dk] M1 val BAcc@10=0.2187 | E5 val BAcc@10=0.2474 | M2 val BAcc@10=0.2706


... (46 outputs left out)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af8685a2160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af8685a2160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

[seed 2209 | epoch 27/30 | 5.1 dk] E5 val BAcc@10=0.2596 (durdu) | M2 val BAcc@10=0.2808 (durdu)
seed 2209 tamamlandı (2.24 saat).


,model,params,best_epoch,val_bacc@10,test_bacc@10,test_top1,test_macro_f1
0,M1,110258,18,0.2519,0.2481,0.1373,0.0118
1,E5,466034,22,0.2644,0.2784,0.1328,0.0182
2,M2,466098,22,0.2821,0.3005,0.1186,0.0272


Karşılaştırma: İP-2'deki M1 test BAcc@10 = 0.2649 | şans düzeyi = 0.20
Not: Model karşılaştırması doğrulama (val) sütununa göre yapılır; test sütunu yalnızca betimseldir.
